# An Introduction to `Causal-Narrative`
**Runtime ~ 20-30 min**

Original paper: ["Mapping Causal Narratives in Political Communication Using Large Language Models"]()

---

This is a tutorial for the package `causal-narrative`. It is a research toolkit for extracting causal narratives from text corpora and constructing **Causal Narrative Networks**. The pipeline is largely unsupervised and consists of the following stages:


1. **Causal Detection**: Identify sentences containing causal relationships (using supervised ML or LLM)
2. **Span Extraction**: Extract cause and effect text spans from causal sentences (using supervised ML or LLM)
3. **Semantic Role Labeling (SRL)**: Convert spans to structured event tuples (Subject-Verb-Object)
4. **Event Clustering**: Group similar events using sentence embeddings
5. **Network Construction**: Build directed graph where nodes are event clusters and edges are causal links
6. **Visualization**: Generate interactive network visualizations

---

In this tutorial, we work with the **Trump Tweet Archive** corpus.

---


## 1. Setup & Installation

Install the package with the desired extras:

```bash
# Core installation
pip install causal-narrative
```

In [4]:
# ---- Environment Setup ----
from pathlib import Path
import warnings
import os

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

# Create output directory
out_dir = Path('output')
out_dir.mkdir(parents=True, exist_ok=True)

In [6]:
# Configure logging with INFO level for detailed tracking
import sys  # <- Add this import to fix NameError
from loguru import logger
logger.remove()
# Add console handler with INFO level
logger.add(sys.stderr, level='INFO', format="<green>{time:HH:mm:ss}</green> | <level>{level: <8}</level> | <level>{message}</level>")
# Add file handler to save all logs
log_file = out_dir / 'tutorial_trump.log'
logger.add(log_file, level='DEBUG', format="{time:YYYY-MM-DD HH:mm:ss} | {level: <8} | {name}:{function}:{line} - {message}")

logger.info("="*80)
logger.info("Starting Causal Narrative Tutorial - Trump Tweet Archive")
logger.info("="*80)

15:31:57 | INFO     | ================================================================================
15:31:57 | INFO     | Starting Causal Narrative Tutorial - Trump Tweet Archive
15:31:57 | INFO     | ================================================================================


In [7]:
# Verify package version
import causal_narrative

logger.info(f'causal-narrative version: {causal_narrative.__version__}')
logger.info(f'Output directory: {out_dir.resolve()}')
logger.info(f'Log file: {log_file.resolve()}')
print(f'causal-narrative version: {causal_narrative.__version__}')
print(f'Output directory: {out_dir.resolve()}')
print(f'Log file: {log_file.resolve()}')

15:31:59 | INFO     | causal-narrative version: 0.1.0
15:31:59 | INFO     | Output directory: /Volumes/Yangdong/causal-narrative/notebook/output
15:31:59 | INFO     | Log file: /Volumes/Yangdong/causal-narrative/notebook/output/tutorial_trump.log


causal-narrative version: 0.1.0
Output directory: /Volumes/Yangdong/causal-narrative/notebook/output
Log file: /Volumes/Yangdong/causal-narrative/notebook/output/tutorial_trump.log


---
## 2. Load the Trump Tweet Archive Dataset

The `datasets` module provides convenient functions for loading built-in datasets.

| Function | Description |
|---|---|
| `list_datasets(verbose=True)` | List all available built-in datasets |
| `get_dataset_info(dataset)` | Get metadata (description, language, available content) for a dataset |
| `load_data(dataset, content, verbose)` | Load a dataset by name; `dataset` selects the corpus, `content` can be `"raw"`, `"sentences"`, or `"srl_res"` |

Available datasets (pass as the `dataset` parameter):

| Dataset Name | Description | Language |
|---|---|---|
| `"trump_tweet_archive"` | Trump Tweet Archives (~32k tweets). Content: `raw`, `sentences`, `srl_res` | English |
| `"state_of_the_union"` | US Presidential State of the Union Addresses (244 speeches, 1790–2018). Content: `raw` | English |

In [4]:
from causal_narrative.datasets import list_datasets, get_dataset_info, load_data

logger.info("Step 2: Loading Trump Tweet Archive Dataset")
logger.info("-" * 80)

# 1. List all available datasets
logger.info("Listing all available datasets...")
list_datasets()

# 2. Inspect dataset metadata
logger.info("Getting dataset metadata for 'trump_tweet_archive'...")
info = get_dataset_info("trump_tweet_archive")
print(f"Description: {info['description']}")
print(f"Language:    {info['language']}")
print(f"Contents:    {list(info['links'].keys())}")
logger.info(f"Dataset info retrieved: {info['description']}")

# 3. Load the raw Trump tweets via load_data (dataset name as parameter)
logger.info("Loading raw Trump tweets from remote...")
df_raw = load_data("trump_tweet_archive", content="raw")

logger.info(f'Successfully loaded {len(df_raw)} tweets')
logger.info(f'DataFrame columns: {list(df_raw.columns)}')
print(f'\nLoaded {len(df_raw)} tweets')
print(f'Columns: {list(df_raw.columns)}')
df_raw.head()

20:21:21 | INFO     | Step 2: Loading Trump Tweet Archive Dataset
20:21:21 | INFO     | --------------------------------------------------------------------------------
20:21:21 | INFO     | Listing all available datasets...
20:21:21 | INFO     | Found 2 available datasets
20:21:21 | INFO     | Getting dataset metadata for 'trump_tweet_archive'...
20:21:21 | INFO     | Dataset info retrieved for: trump_tweet_archive
20:21:21 | INFO     | Dataset info retrieved: Tweets from the Trump Tweet Archives (https://www.thetrumparchive.com/)
20:21:21 | INFO     | Loading raw Trump tweets from remote...
20:21:21 | INFO     | Loading dataset: trump_tweet_archive/raw from https://www.dropbox.com/s/lxqz454n29iqktn/trump_archive.csv?dl=1
20:21:21 | INFO     | Loading trump_tweet_archive/raw from remote...


Available Datasets:

1. trump_tweet_archive
   - Description: Tweets from the Trump Tweet Archives (https://www.thetrumparchive.com/)
   - Language: english
   - Available content: raw, sentences, srl_res

2. state_of_the_union
   - Description: State of the Union Addresses by US Presidents (1790-2018), sourced from the American Presidency Project
   - Language: english
   - Available content: raw

Description: Tweets from the Trump Tweet Archives (https://www.thetrumparchive.com/)
Language:    english
Contents:    ['raw', 'sentences', 'srl_res']


20:21:25 | INFO     | Successfully loaded DataFrame with 32323 rows
20:21:25 | INFO     | Loaded DataFrame with 32323 rows
20:21:25 | INFO     | Successfully loaded 32323 tweets
20:21:25 | INFO     | DataFrame columns: ['id', 'doc']



Loaded 32323 tweets
Columns: ['id', 'doc']


,id,doc
0,9.845497e+16,Republicans and Democrats have both created ou...
1,1.234653e+18,I was thrilled to be back in the Great city of...
2,1.304875e+18,The Unsolicited Mail In Ballot Scam is a major...
3,1.223641e+18,Getting a little exercise this morning!
4,1.215248e+18,Thank you Elise!


In [ ]:
# Filter out very short tweets
df = df_raw[df_raw['doc'].str.len() > 20].copy().reset_index(drop=True)

#df = df.sample(n=2000, random_state=42)

print(f'Original:  {len(df_raw)} tweets')
print(f'Filtered:  {len(df)} tweets (len > 20 chars)')
print(f'\nSample tweets:')
for i, (_, row) in enumerate(df.head(5).iterrows()):
    print(f'  [{i}] {row["doc"][:100]}...' if len(row['doc']) > 100 else f'  [{i}] {row["doc"]}')

Original:  32323 tweets
Filtered:  2000 tweets (len > 20 chars)

Sample tweets:
  [0] I made a great deal of money in Atlantic City, but left years ago when I saw so many political mista...
  [1] Both Commiecast MSNBC &amp, Fake News CNN are watching their Ratings TANK. Fredo on CNN is dying. Do...
  [2] The Washington Post and CNN have typically written false stories about our trade negotiations with C...
  [3] Housing prices will be going up big league--a great time to buy--good luck!
  [4] I am now inspecting the Old Post Office on Pennsylvania Avenue - will be a great hotel. Soon off to ...


---
## 3. Sentence Segmentation

The `segmentation` module provides functions to split documents into sentences.

### Functions

| Function | Description | Key Parameters |
|----------|-------------|----------------|
| `segment_text(text, language, min_length, max_length)` | Sentence tokenization with optional filtering | `language`: 'english' (default), `min_length`/`max_length`: optional |
| `normalize_text(text, remove_extra_whitespace)` | Text normalization | `remove_extra_whitespace`: bool (True) |
| `ensure_nltk_data()` | Download NLTK punkt data if missing | No parameters |

In [8]:
from causal_narrative.segmentation import (
    segment_text,
    normalize_text,
    ensure_nltk_data,
)

logger.info("Step 3: Preparing Text Segmentation")
logger.info("-" * 80)

# Ensure NLTK data is available (will auto-download if missing)
logger.info("Checking NLTK punkt tokenizer data...")
try:
    ensure_nltk_data()
    logger.info("✓ NLTK data ready for sentence segmentation")
except Exception as e:
    logger.error(f"✗ Failed to ensure NLTK data: {e}")
    raise

15:32:09 | INFO     | Step 3: Preparing Text Segmentation
15:32:09 | INFO     | --------------------------------------------------------------------------------
15:32:09 | INFO     | Checking NLTK punkt tokenizer data...
15:32:12 | INFO     | ✓ NLTK data ready for sentence segmentation


In [20]:
import pandas as pd
from tqdm import tqdm

# Segment all tweets into sentences
sentences = []
doc_ids = []
sent_ids = []

for idx, row in tqdm(df.iterrows(), total=len(df), desc='Segmenting'):
    sents = segment_text(row['doc'], language='english', min_length=10, max_length=600)
    for sent_idx, sent in enumerate(sents):
        sentences.append(sent)
        doc_ids.append(idx)
        sent_ids.append(sent_idx)

df_sentences = pd.DataFrame({
    'doc_id': doc_ids,
    'sent_id': sent_ids,
    'sentence': sentences,
})

print(f'\nExtracted {len(df_sentences)} sentences from {len(df)} tweets')
print(f'Average sentences per tweet: {len(df_sentences)/len(df):.1f}')
df_sentences.head(10)

Segmenting: 100%|██████████| 2000/2000 [00:00<00:00, 27395.40it/s]


Extracted 3958 sentences from 2000 tweets
Average sentences per tweet: 2.0


,doc_id,sent_id,sentence
0,13868,0,"I made a great deal of money in Atlantic City,..."
1,13868,1,I have ZERO involvement!
2,25940,0,"Both Commiecast MSNBC &amp, Fake News CNN are ..."
3,25940,1,Fredo on CNN is dying.
4,25940,2,Dont know why FoxNews wants to be more like them?
5,25940,3,Theyll all die together as other outlets take ...
6,25940,4,Only pro Trump Fox shows do well.
7,25940,5,Rest are nothing.
8,25940,6,Hows Shep doing?
9,25101,0,The Washington Post and CNN have typically wri...


---
## 4. Causal Relationship Detection

The `CausalDetector` class provides a **unified interface** for detecting causal relationships.
It supports three backends:

| Method | Speed | Accuracy | Requirements | Parallel |
|--------|-------|----------|-------------|----------|
| `'heuristic'` | Very fast | Moderate | None | N/A |
| `'llm'` | Slow | High | API key | Yes (multi-key) |
| `'bert'` | Medium | High | HuggingFace model | Batch |

### `CausalDetector.__init__()` Parameters

| Parameter | Type | Default | Description |
|-----------|------|---------|-------------|
| `method` | str | `'heuristic'` | `'heuristic'`, `'llm'`, or `'bert'` |
| `causal_patterns` | List[str] | `DEFAULT_CAUSAL_PATTERNS` | Custom regex patterns (heuristic) |
| `api_keys` | List[str] | None | API keys for LLM (required for `'llm'`) |
| `model_name` | str | `'deepseek-ai/DeepSeek-V3'` | LLM model identifier |
| `base_url` | str | `'https://api.siliconflow.cn/v1'` | API endpoint URL |
| `max_workers` | int | 4 | Concurrent API workers (LLM) |
| `timeout` | int | 60 | API timeout in seconds |
| `max_retries` | int | 3 | Max retry attempts |
| `temperature` | float | 0.0 | LLM sampling temperature |
| `system_prompt` | str | None | Custom system prompt (LLM) |
| `user_template` | str | None | Custom user template with `{sentence}` (LLM) |
| `bert_model_name` | str | `'causal-narrative/roberta-causal-narrative-classifier'` | HuggingFace model name (BERT); auto-set if omitted |
| `bert_model_path` | str | None | Local model path (takes precedence if exists) |
| `bert_cache_dir` | str | `'model'` | Directory to cache downloaded model for offline reuse |
| `bert_device` | str | None | Device: `'cuda'`, `'cpu'`, or auto |
| `bert_threshold` | float | 0.5 | Classification threshold (BERT) |

### Methods

| Method | Description | Returns |
|--------|-------------|----------|
| `detect(sentences, show_progress, batch_size)` | Batch detection | `List[DetectionResult]` |
| `get_cost_summary()` | LLM cost stats | `Dict` or `None` |

### `DetectionResult` Fields

| Field | Type | Description |
|-------|------|-------------|
| `has_causality` | bool | Whether causal relationship was detected |
| `score` | float | Confidence score [0.0, 1.0] |
| `rationale` | str | Explanation for the decision |
| `causal_type` | str | Type: direct, enabling, conditional, etc. |
| `method` | str | Backend used: 'heuristic', 'llm', 'bert' |

In [21]:
from causal_narrative.detection import (
    CausalDetector,
    DetectionResult,
)

detector = CausalDetector(method='bert')

detection_results = detector.detect(
    df_sentences['sentence'].tolist(),
    show_progress=True,
)

20:51:26 | INFO     | Loading BERT model from local cache: model/causal-narrative_roberta-causal-narrative-classifier
Loading weights: 100%|██████████| 201/201 [00:00<00:00, 2572.66it/s, Materializing param=roberta.encoder.layer.11.output.dense.weight]              
20:51:26 | INFO     | BERTCausalDetector loaded from: model/causal-narrative_roberta-causal-narrative-classifier
20:51:26 | INFO     | Device: cpu
20:51:26 | INFO     | CausalDetector initialized with BERT method (model=causal-narrative/roberta-causal-narrative-classifier, cache_dir=model)
20:51:26 | INFO     | Starting batch BERT detection for 3958 sentences
BERT detection: 100%|██████████| 3958/3958 [02:12<00:00, 29.96sent/s]
20:53:38 | INFO     | Completed batch BERT detection: 615 causal sentences found


In [22]:
# Add results to DataFrame
df_sentences['has_causality'] = [r.has_causality for r in detection_results]
df_sentences['causal_score'] = [r.score for r in detection_results]
df_sentences['causal_type'] = [r.causal_type for r in detection_results]
df_sentences['detection_method'] = [r.method for r in detection_results]

# Filter causal sentences
df_causal = df_sentences[df_sentences['has_causality']].copy().reset_index(drop=True)

# Save df_causal to output directory
df_causal.to_pickle(str(out_dir / "detect_results.pkl"))

print(f'\nTotal sentences:  {len(df_sentences)}')
print(f'Causal sentences: {len(df_causal)} ({100*len(df_causal)/len(df_sentences):.1f}%)')
print(f'\nSample causal sentences:')
for _, row in df_causal.head(5).iterrows():
    print(f'  (score={row["causal_score"]:.2f}) {row["sentence"][:100]}')


Total sentences:  3958
Causal sentences: 615 (15.5%)

Sample causal sentences:
  (score=0.89) I made a great deal of money in Atlantic City, but left years ago when I saw so many political mista
  (score=0.99) Theyll all die together as other outlets take their place.
  (score=0.60) I feel sorry for Rosie 's new partner in love whose parents are devastated at the thought of their d
  (score=0.99) If you dont deliver the goods, people will eventually catch on.
  (score=0.67) The Stock Market has been creating tremendous benefits for our country in the form of not only Recor


---
## 5. Cause/Effect Span Extraction

The `CausalSpanExtractor` extracts the specific cause and effect text spans from causal sentences.

### `CausalSpanExtractor.__init__()` Parameters

| Parameter | Type | Default | Description |
|-----------|------|---------|-------------|
| `method` | str | `'pattern'` | `'pattern'`, `'llm'`, or `'bert'` |
| `span_patterns` | List[tuple] | `DEFAULT_SPAN_PATTERNS` | Custom (regex, order) patterns |
| `api_keys` | List[str] | None | API keys for LLM |
| `model_name` | str | ... | LLM model |
| `base_url` | str | ... | API endpoint |
| `max_workers` | int | 4 | Concurrent workers |
| `system_prompt` | str | ... | Custom system prompt |
| `user_template` | str | ... | Custom user template |
| `bert_model_name` | str | `'causal-narrative/roberta-causal-span-extractor'` | HF model for BERT span extraction; auto-set if omitted |
| `bert_model_path` | str | None | Local model path (takes precedence if exists) |
| `bert_cache_dir` | str | `'model'` | Directory to cache downloaded model for offline reuse |
| `bert_device` | str | None | Device: `'cuda'`, `'cpu'`, or auto |

### Methods

| Method | Description | Returns |
|--------|-------------|----------|
| `extract(sentences, show_progress)` | Batch extraction | `List[Optional[SpanExtractionResult]]` |

### `SpanExtractionResult` Fields

| Field | Type | Description |
|-------|------|-------------|
| `cause_text` | str | Extracted cause text span |
| `effect_text` | str | Extracted effect text span |
| `cause_start` / `cause_end` | int | Char-level cause span indices |
| `effect_start` / `effect_end` | int | Char-level effect span indices |
| `confidence` | float | Extraction confidence [0.0, 1.0] |
| `method` | str | Backend used: 'pattern', 'llm', 'bert' |

In [ ]:
from causal_narrative.extraction import (
    CausalSpanExtractor,
    SpanExtractionResult,
)

# LLM-based span extraction uses its own prompt templates:
from causal_narrative.extraction import EXTRACTION_SYSTEM_PROMPT, EXTRACTION_USER_TEMPLATE

print('=== Span Extraction System Prompt ===')
print(EXTRACTION_SYSTEM_PROMPT)

print('\n=== Span Extraction User Template ===')
print(EXTRACTION_USER_TEMPLATE)


extractor = CausalSpanExtractor(
    method='llm',
    api_keys=[
    "sk-xxxx",
    "sk-yyyy",
    ],  # Multiple keys for parallelism
    model_name='MODEL_NAME',                   # e.g. 'deepseek-ai/DeepSeek-V3', 'gpt-4o-mini'
    base_url='BASE_URL',                   # e.g. 'https://api.openai.com/v1'
    #max_workers=4,                                  # Parallel workers (capped at #keys)
    timeout=60,                                     # Request timeout
    temperature=0.0,                                # Deterministic output
)



# OR use BERT model

# extractor = CausalSpanExtractor(method='bert')

20:59:11 | INFO     | Initialized LLM client with 10 keys, max_workers=10, model=deepseek-ai/DeepSeek-V3
20:59:11 | INFO     | CausalSpanExtractor initialized with LLM method (base_url=https://api.siliconflow.cn/v1)
20:59:11 | INFO     |   API keys: 10, max_workers: 10 (auto-set to #keys) - you can customize max_workers to limit concurrency


=== Span Extraction System Prompt ===
You are an expert in extracting causal spans from text.

Your task is to identify the specific text spans that represent the CAUSE and the EFFECT in a causal sentence.

Guidelines:
- Extract the minimal span that captures the complete cause/effect
- Include necessary modifiers but avoid unnecessary words
- Use exact character positions (start index is inclusive, end index is exclusive)
- The cause span should represent what leads to the effect
- The effect span should represent what results from the cause

Respond ONLY with valid JSON. Do not include any markdown formatting or code blocks.

=== Span Extraction User Template ===
Extract cause and effect spans from this sentence:

Sentence: "{sentence}"

Respond in this exact JSON format:
{{
  "cause_text": "extracted cause span",
  "effect_text": "extracted effect span",
  "cause_start": 0,
  "cause_end": 10,
  "effect_start": 15,
  "effect_end": 30
}}

Character positions should be 0-indexed. Start

In [27]:
span_results = extractor.extract(
    df_causal['sentence'].tolist(),
    show_progress=True,
)

21:00:04 | INFO     | Starting batch LLM extraction for 615 sentences
21:00:04 | INFO     | Starting batch LLM generation for 615 prompts using 10 workers
Generating: 100%|██████████| 615/615 [04:26<00:00,  2.31it/s]
21:04:31 | INFO     | Completed batch LLM generation: 615 prompts, 0 errors, 184441 total tokens, cost=$0.0315
21:04:31 | INFO     | Completed batch LLM extraction: 615/615 successful, 0 fallbacks


In [30]:
# Add results to DataFrame
df_causal['cause_span'] = [r.cause_text if r else '' for r in span_results]
df_causal['effect_span'] = [r.effect_text if r else '' for r in span_results]
df_causal['span_confidence'] = [r.confidence if r else 0.0 for r in span_results]
df_causal['span_method'] = [r.method if r else 'none' for r in span_results]

# Filter out sentences where extraction failed
df_spans = df_causal[(df_causal['cause_span'] != '') & (df_causal['effect_span'] != '')].copy()
df_spans = df_spans.reset_index(drop=True)

# Save df_spans to out_dir as spans_results.pkl
import os
out_path = os.path.join(out_dir, 'spans_results.pkl')
df_spans.to_pickle(out_path)

print(f'\nCausal sentences:        {len(df_causal)}')
print(f'Successfully extracted:   {len(df_spans)} ({100*len(df_spans)/max(len(df_causal),1):.1f}%)')
print(f'Span results saved to:    {out_path}')
print(f'\nSample cause-effect pairs:')
for _, row in df_spans.head(5).iterrows():
    print(f'  CAUSE:  {row["cause_span"][:60]}')
    print(f'  EFFECT: {row["effect_span"][:60]}')
    print()


Causal sentences:        615
Successfully extracted:   608 (98.9%)
Span results saved to:    output/spans_results.pkl

Sample cause-effect pairs:
  CAUSE:  so many political mistakes being made
  EFFECT: left years ago

  CAUSE:  other outlets take their place
  EFFECT: Theyll all die together

  CAUSE:  their daughter being with Rosie--a true loser
  EFFECT: parents are devastated

  CAUSE:  dont deliver the goods
  EFFECT: people will eventually catch on

  CAUSE:  The Stock Market
  EFFECT: tremendous benefits



---
## 6. Semantic Role Labeling (SRL)

SRL extracts structured **Argument Roles (ARG0, V, ARG1)** from cause/effect text spans.

Both spaCy and AllenNLP methods output consistent dictionary structures compatible with downstream event clustering.

### Available Methods

| Method | Class | Speed | Quality | Requirements |
|---------|-------|-------|---------|--------------|
| spaCy | `SpacySRL` | Fast | Good | `en_core_web_sm` |
| AllenNLP | `AllenNLPSRL` | Slow | Best | `allennlp` + Python 3.9-3.10 |

### `SpacySRL.__init__()` Parameters

| Parameter | Type | Default | Description |
|-----------|------|---------|-------------|
| `model_name` | str | `'en_core_web_sm'` | spaCy model name |

### Main Methods

| Method | Description | Returns |
|--------|-------------|----------|
| `process(texts, batch_size)` | Process multiple texts | `List[Dict[str, Any]]` |

### SRL Output Format

Each result is a dictionary with:
- `words`: List of tokens
- `verbs`: List of verb frames, each containing:
  - `verb`: The verb text
  - `description`: Formatted as "[ARG0: ...] [V: ...] [ARG1: ...]"
  - `tags`: BIO tags for each word

### Utility Functions

| Function | Description | Returns |
|----------|-------------|---------|
| `extract_roles(srl_result)` | Extract ARG0, V, ARG1 from SRL result | `Dict[str, Optional[str]]` |
| `get_srl(method, **kwargs)` | Factory function to get SRL processor | `SpacySRL` or `AllenNLPSRL` |
| `is_allennlp_available()` | Check if AllenNLP is available | `bool` |

In [4]:
from causal_narrative.semantic_role_labeling import (
    get_srl, 
    is_allennlp_available, 
    extract_roles,
    SpacySRL,
    AllenNLPSRL
)

is_allennlp_available()

False

In [6]:
import pandas as pd
import os

df_spans = pd.read_pickle(os.path.join(out_dir, 'spans_results.pkl'))

In [7]:
# Initialize spaCy SRL
srl = get_srl('spacy', model_name='en_core_web_sm')

cause_srl_results = srl.process(df_spans['cause_span'].tolist(), batch_size=32, show_progress=True)
effect_srl_results = srl.process(df_spans['effect_span'].tolist(), batch_size=32, show_progress=True)

21:18:22 | WARNING  | spaCy model 'en_core_web_sm' not found
21:18:22 | INFO     | Attempting to download spaCy model 'en_core_web_sm'...
21:18:22 | INFO     | This is a one-time download and may take a few moments...
21:18:22 | INFO     | Running: python -m spacy download en_core_web_sm
21:18:28 | INFO     | ✓ spaCy model 'en_core_web_sm' downloaded successfully
21:18:30 | INFO     | ✓ spaCy model 'en_core_web_sm' loaded successfully
21:18:30 | INFO     | Starting batch SRL processing for 608 texts using spaCy
Processing SRL (spaCy): 100%|██████████| 608/608 [00:00<00:00, 815.89texts/s]
21:18:30 | INFO     | Completed batch SRL processing: 400/608 texts with verbs found
21:18:30 | INFO     | Starting batch SRL processing for 608 texts using spaCy
Processing SRL (spaCy): 100%|██████████| 608/608 [00:00<00:00, 867.98texts/s]
21:18:31 | INFO     | Completed batch SRL processing: 507/608 texts with verbs found


In [9]:
import json

df_spans['cause_srl'] = [json.dumps(r) for r in cause_srl_results]
df_spans['effect_srl'] = [json.dumps(r) for r in effect_srl_results]

df_spans.to_pickle(os.path.join(out_dir, 'srl_results.pkl'))

In [10]:
# Count successful extractions
cause_success = sum(1 for r in cause_srl_results if r.get('verbs'))
effect_success = sum(1 for r in effect_srl_results if r.get('verbs'))

print(f'Cause SRL success:  {cause_success}/{len(df_spans)} ({100*cause_success/max(len(df_spans),1):.1f}%)')
print(f'Effect SRL success: {effect_success}/{len(df_spans)} ({100*effect_success/max(len(df_spans),1):.1f}%)')

# Show some examples
print('\nSample SRL results:\n')
for i in range(min(3, len(df_spans))):
    print(f'  [{i}] Cause: "{df_spans.iloc[i]["cause_span"][:50]}"')
    cause_roles = extract_roles(cause_srl_results[i])
    if cause_roles['V']:
        print(f'       SRL: ARG0={cause_roles["ARG0"]}, V={cause_roles["V"]}, ARG1={cause_roles["ARG1"]}')
    
    print(f'      Effect: "{df_spans.iloc[i]["effect_span"][:50]}"')
    effect_roles = extract_roles(effect_srl_results[i])
    if effect_roles['V']:
        print(f'       SRL: ARG0={effect_roles["ARG0"]}, V={effect_roles["V"]}, ARG1={effect_roles["ARG1"]}')
    print()

Cause SRL success:  400/608 (65.8%)
Effect SRL success: 507/608 (83.4%)

Sample SRL results:

  [0] Cause: "so many political mistakes being made"
       SRL: ARG0=None, V=make, ARG1=so many political mistakes
      Effect: "left years ago"
       SRL: ARG0=None, V=leave, ARG1=None

  [1] Cause: "other outlets take their place"
       SRL: ARG0=other outlets, V=take, ARG1=their place
      Effect: "Theyll all die together"
       SRL: ARG0=They, V=die, ARG1=None

  [2] Cause: "their daughter being with Rosie--a true loser"
       SRL: ARG0=their daughter, V=be, ARG1=None
      Effect: "parents are devastated"
       SRL: ARG0=None, V=devastate, ARG1=parents



---
## 7. Event Embedding & Clustering

This is the core analytical step. Events are embedded and clustered to discover recurrent narrative patterns.

### 7.1 Embedding

The `SentenceEmbedder` class wraps Sentence-Transformers models.

| Parameter | Type | Default | Description |
|-----------|------|---------|-------------|
| `model_name` | str | `'all-MiniLM-L6-v2'` | Sentence-Transformers model |
| `cache_dir` | str | None | Model cache directory |
| `device` | str | None | Device: 'cuda', 'cpu', or auto |

| Method | Description |
|--------|-------------|
| `embed(texts, batch_size, show_progress, normalize)` | Embed texts |
| `get_embedding_dim()` | Get embedding dimensionality |
| `compute_similarity(emb1, emb2)` | Cosine similarity matrix |
| `find_nearest_neighbors(query, candidates, top_k)` | Find closest embeddings |

In [13]:
from causal_narrative.embedding import (
    SentenceEmbedder,
    load_embedder,
    generate_role_based_embeddings,
    generate_phrase_embeddings,
    role_based_pca,
    phrase_pca,
)

import numpy as np

# Initialize embedder
embedder = SentenceEmbedder(model_name='all-MiniLM-L6-v2')

print(f'Model: all-MiniLM-L6-v2')
print(f'Embedding dim: {embedder.get_embedding_dim()}')

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2077.25it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
22:59:26 | INFO     | Loaded embedding model: all-MiniLM-L6-v2 (dimension: 384)


Model: all-MiniLM-L6-v2
Embedding dim: 384


In [15]:
from causal_narrative.semantic_role_labeling import is_event_srl

df = df_srl.copy()
df['cause_valid_for_role'] = df['cause_srl'].apply(is_event_srl)
df['effect_valid_for_role'] = df['effect_srl'].apply(is_event_srl)

# Statistics
print('=' * 60)
print('SRL Validity Statistics')
print('=' * 60)
print(f'Total samples: {len(df)}')
print(f'\nCause:')
print(f'  - Role-based (valid SRL): {df["cause_valid_for_role"].sum()}')
print(f'  - Phrase-based (invalid SRL): {(~df["cause_valid_for_role"]).sum()}')
print(f'\nEffect:')
print(f'  - Role-based (valid SRL): {df["effect_valid_for_role"].sum()}')
print(f'  - Phrase-based (invalid SRL): {(~df["effect_valid_for_role"]).sum()}')

SRL Validity Statistics
Total samples: 7687

Cause:
  - Role-based (valid SRL): 3600
  - Phrase-based (invalid SRL): 4087

Effect:
  - Role-based (valid SRL): 3526
  - Phrase-based (invalid SRL): 4161


In [20]:
df['cause_span'] = df['cause_text']
df['effect_span'] = df['effect_text']

In [ ]:
# ===== Initialize Clustering Result Columns =====
df['cause_srl_cluster_id'] = -1
df['cause_srl_event'] = ''
df['cause_phrase_cluster_id'] = -1
df['cause_phrase_event'] = ''
df['effect_srl_cluster_id'] = -1
df['effect_srl_event'] = ''
df['effect_phrase_cluster_id'] = -1
df['effect_phrase_event'] = ''

print('✓ Clusteringresult columnsInitialize')

In [22]:
# ===== Cause Clustering =====
print('\n' + '=' * 70)
print('Cause Clustering（Role-based + Phrase-based）')
print('=' * 70)


Cause 聚类（Role-based + Phrase-based）


In [23]:
import json

# ========== 1. Role-based Clustering（Valid SRL） ==========
print(f'\n[1/2] Role-based Clustering')

# Prepare data
cause_role_mask = df['cause_valid_for_role'].values
cause_role_df = df[cause_role_mask].copy()

cause_srl_dicts = [json.loads(s) if isinstance(s, str) else s for s in cause_role_df['cause_srl']]

# Embedding
print('  → Generate Role-based embeddings...')
embeddings_cause_role = generate_role_based_embeddings(
    srl_results=cause_srl_dicts,
    embedder=embedder,
    batch_size=64,
    show_progress=False
)

# PCA Dimensionality reduction
print('  → PCA Dimensionality reduction (ARG0:10, V:10, ARG1:10)...')
embeddings_cause_role_pca, pca_models_cause_role = role_based_pca(
    embeddings_cause_role,
    dim_a0=30,
    dim_v=30,
    dim_a1=30,
    random_state=42
)

# ========== 2. Phrase-based Clustering（Invalid SRL） ==========
print(f'\n[2/2] Phrase-based Clustering')

# Prepare data
cause_phrase_mask = ~df['cause_valid_for_role'].values
cause_phrase_df = df[cause_phrase_mask].copy()

cause_texts = cause_phrase_df['cause_span'].tolist()

# Embedding
print('  → Generate Phrase embeddings...')
embeddings_cause_phrase = generate_phrase_embeddings(
    texts=cause_texts,
    embedder=embedder,
    batch_size=64,
    show_progress=False
)

# PCA Dimensionality reduction
print('  → PCA Dimensionality reduction (dim=30)...')
embeddings_cause_phrase_pca, pca_model_cause_phrase = phrase_pca(
    embeddings_cause_phrase,
    n_components=90,
    random_state=42
)

23:00:29 | INFO     | Generating role-based embeddings for 3600 SRL results



[1/2] Role-based 聚类
  → 生成 Role-based embeddings...


23:00:30 | INFO     | Role-based embeddings generated: shape=(3600, 1152)
23:00:30 | INFO     | Applying role-based PCA: A0(384->30), V(384->30), A1(384->30)
23:00:30 | INFO     | Role-based PCA completed: output shape=(3600, 90), variance retained: A0=0.784, V=0.594, A1=0.497
23:00:30 | INFO     | Generating phrase embeddings for 4087 texts


  → PCA 降维 (ARG0:10, V:10, ARG1:10)...

[2/2] Phrase-based 聚类
  → 生成 Phrase embeddings...


23:00:31 | INFO     | Phrase embeddings generated: shape=(4087, 384)
23:00:31 | INFO     | Applying phrase PCA: 384 -> 90
23:00:31 | INFO     | Phrase PCA completed: output shape=(4087, 90), variance retained=0.719


  → PCA 降维 (dim=30)...


In [24]:
import json

# ===== Effect Clustering =====
print('\n' + '=' * 70)
print('Effect Clustering（Role-based + Phrase-based）')
print('=' * 70)

# ========== 1. Role-based Clustering（Valid SRL） ==========
print(f'\n[1/2] Role-based Clustering')

# Prepare data
effect_role_mask = df['effect_valid_for_role'].values
effect_role_df = df[effect_role_mask].copy()

effect_srl_dicts = [json.loads(s) if isinstance(s, str) else s for s in effect_role_df['effect_srl']]

# Embedding
print('  → Generate Role-based embeddings...')
embeddings_effect_role = generate_role_based_embeddings(
    srl_results=effect_srl_dicts,
    embedder=embedder,
    batch_size=64,
    show_progress=False
)

# PCA Dimensionality reduction
print('  → PCA Dimensionality reduction (ARG0:10, V:10, ARG1:10)...')
embeddings_effect_role_pca, pca_models_effect_role = role_based_pca(
    embeddings_effect_role,
    dim_a0=30,
    dim_v=30,
    dim_a1=30,
    random_state=42
)

# ========== 2. Phrase-based Clustering（Invalid SRL） ==========
print(f'\n[2/2] Phrase-based Clustering')

# Prepare data
effect_phrase_mask = ~df['effect_valid_for_role'].values
effect_phrase_df = df[effect_phrase_mask].copy()


effect_texts = effect_phrase_df['effect_span'].tolist()

# Embedding
print('  → Generate Phrase embeddings...')
embeddings_effect_phrase = generate_phrase_embeddings(
    texts=effect_texts,
    embedder=embedder,
    batch_size=64,
    show_progress=False
)

# PCA Dimensionality reduction
print('  → PCA Dimensionality reduction (dim=30)...')
embeddings_effect_phrase_pca, pca_model_effect_phrase = phrase_pca(
    embeddings_effect_phrase,
    n_components=90,
    random_state=42
)

23:00:37 | INFO     | Generating role-based embeddings for 3526 SRL results



Effect 聚类（Role-based + Phrase-based）

[1/2] Role-based 聚类
  → 生成 Role-based embeddings...


23:00:38 | INFO     | Role-based embeddings generated: shape=(3526, 1152)
23:00:38 | INFO     | Applying role-based PCA: A0(384->30), V(384->30), A1(384->30)
23:00:38 | INFO     | Role-based PCA completed: output shape=(3526, 90), variance retained: A0=0.818, V=0.597, A1=0.513
23:00:38 | INFO     | Generating phrase embeddings for 4161 texts


  → PCA 降维 (ARG0:10, V:10, ARG1:10)...

[2/2] Phrase-based 聚类
  → 生成 Phrase embeddings...


23:00:39 | INFO     | Phrase embeddings generated: shape=(4161, 384)
23:00:39 | INFO     | Applying phrase PCA: 384 -> 90
23:00:39 | INFO     | Phrase PCA completed: output shape=(4161, 90), variance retained=0.708


  → PCA 降维 (dim=30)...


### 7.2 EventClusterer

The `EventClusterer` is the main clustering class. It supports:

| Clustering Method | Description |
|-------------------|-------------|
| `'role_based'` | **Role-based Event Embedding**: separately embeds ARG0/V/ARG1, concatenates, then PCA+cluster |
| `'phrase'` | **Phrase-based**: embeds raw phrases directly, PCA+cluster |

| Algorithm | Description | Key Param |
|-----------|-------------|----------|
| `'dpmeans'` | Auto-determines cluster count | `delta`: distance threshold |
| `'kmeans'` | Fixed cluster count | `n_clusters`: number of clusters |
| `'hdbscan'` | Density-based | `min_cluster_size`: min points |

### `EventClusterer.__init__()` Parameters

| Parameter | Type | Default | Description |
|-----------|------|---------|-------------|
| `method` | str | `'role_based'` | `'role_based'` or `'phrase'` |
| `algorithm` | str | `'dpmeans'` | `'dpmeans'`, `'kmeans'`, `'hdbscan'` |
| `model_name` | str | `'all-MiniLM-L6-v2'` | Sentence embedding model |
| `dim_a0` | int | 10 | PCA dim for ARG0 role (role_based only) |
| `dim_v` | int | 5 | PCA dim for V role (role_based only) |
| `dim_a1` | int | 10 | PCA dim for ARG1 role (role_based only) |
| `phrase_pca_dim` | int | 15 | PCA dim for phrase-based method |
| `delta` | float | 0.2 | DP-Means distance threshold |
| `batch_size` | int | 100 | DP-Means mini-batch size |
| `n_clusters` | int | None | Number of clusters (kmeans) |
| `min_cluster_size` | int | 5 | Min cluster size |
| `random_state` | int | 42 | Random seed |

### `ClusterInfo` Fields

| Field | Type | Description |
|-------|------|-------------|
| `cluster_id` | int | Cluster ID |
| `name` | str | Representative name (most common text) |
| `size` | int | Number of events |
| `texts` | List[str] | All texts in cluster |
| `text_counts` | Counter | Text frequency counts |

In [25]:
from causal_narrative.event_clustering import (
    EventClusterer,
    ClusteringConfig,
    ClusterInfo,
    ClusteringResult,
    run_dpmeans,
    run_kmeans,
    run_hdbscan,
    count_cluster_sizes,
    generate_cluster_names_from_srl,
    generate_cluster_names_from_texts,
)

# ========== 1. Role-based Clustering ==========
print(f'\n[1/2] Role-based Clustering')

# Clustering
print('  → K-Means clustering (k=100)...')
cluster_ids_cause_role, clustering_model_cause_role = run_kmeans(
    embeddings_cause_role_pca,
    n_clusters=100,
    random_state=42
)

# cluster_ids_cause_role, clustering_model_cause_role = run_hdbscan(
#     embeddings_cause_role_pca,
#     min_cluster_size=5,
#     min_samples=3,
#     metric='cosine'
# )

# cluster_ids, clustering_model = run_dpmeans(
#     reduced_embs,
#     delta=0.1,
#     batch_size=50,
#     random_state=42
# )


# Generate cluster names
print('  → Generating cluster names...')
event_names_cause_role = generate_cluster_names_from_srl(
    labels=cluster_ids_cause_role,
    srl_results=cause_srl_dicts,
    fallback_texts=cause_role_df['cause_span'].tolist()
)

# Save results to original DataFrame
df.loc[cause_role_df.index, 'cause_srl_cluster_id'] = cluster_ids_cause_role
df.loc[cause_role_df.index, 'cause_srl_event'] = [event_names_cause_role[cid] for cid in cluster_ids_cause_role]

print(f'  ✓ Completed: {len(event_names_cause_role)} clusters')

# ========== 2. Phrase-based Clustering ==========
print(f'\n[2/2] Phrase-based Clustering')

# Clustering
print('  → K-Means clustering (k=100)...')
cluster_ids_cause_phrase, clustering_model_cause_phrase = run_kmeans(
    embeddings_cause_phrase_pca,
    n_clusters=100,
    random_state=42
)

# Generate cluster names
print('  → Generating cluster names...')
event_names_cause_phrase = generate_cluster_names_from_texts(
    labels=cluster_ids_cause_phrase,
    texts=cause_texts
)

# Save results to original DataFrame
df.loc[cause_phrase_df.index, 'cause_phrase_cluster_id'] = cluster_ids_cause_phrase
df.loc[cause_phrase_df.index, 'cause_phrase_event'] = [event_names_cause_phrase[cid] for cid in cluster_ids_cause_phrase]

print(f'  ✓ Completed: {len(event_names_cause_phrase)} clusters')

print('\n' + '=' * 70)
print('✓ Cause Clustering Completed')
print('=' * 70)


[1/2] Role-based Clustering
  → K-Means clustering (k=100)...
    K-Means (k=100, n=3600)...
      ✓ 100 clusters
  → Generating cluster names...
  ✓ Completed: 100 clusters

[2/2] Phrase-based Clustering
  → K-Means clustering (k=100)...
    K-Means (k=100, n=4087)...
      ✓ 100 clusters
  → Generating cluster names...
  ✓ Completed: 100 clusters

✓ Cause Clustering Completed


In [26]:
# ========== 1. Role-based Clustering ==========
print(f'\n[1/2] Role-based Clustering')

# Clustering
print('  → K-Means clustering (k=100)...')
cluster_ids_effect_role, clustering_model_effect_role = run_kmeans(
    embeddings_effect_role_pca,
    n_clusters=100,
    random_state=42
)

# Generate cluster names
print('  → Generating cluster names...')
event_names_effect_role = generate_cluster_names_from_srl(
    labels=cluster_ids_effect_role,
    srl_results=effect_srl_dicts,
    fallback_texts=effect_role_df['effect_span'].tolist()
)

# Save results to original DataFrame
df.loc[effect_role_df.index, 'effect_srl_cluster_id'] = cluster_ids_effect_role
df.loc[effect_role_df.index, 'effect_srl_event'] = [event_names_effect_role[cid] for cid in cluster_ids_effect_role]

print(f'  ✓ Completed: {len(event_names_effect_role)} clusters')

# ========== 2. Phrase-based Clustering ==========
print(f'\n[2/2] Phrase-based Clustering')

# Clustering
print('  → K-Means clustering (k=100)...')
cluster_ids_effect_phrase, clustering_model_effect_phrase = run_kmeans(
    embeddings_effect_phrase_pca,
    n_clusters=100,
    random_state=42
)

# Generate cluster names
print('  → Generating cluster names...')
event_names_effect_phrase = generate_cluster_names_from_texts(
    labels=cluster_ids_effect_phrase,
    texts=effect_texts
)

# Save results to original DataFrame
df.loc[effect_phrase_df.index, 'effect_phrase_cluster_id'] = cluster_ids_effect_phrase
df.loc[effect_phrase_df.index, 'effect_phrase_event'] = [event_names_effect_phrase[cid] for cid in cluster_ids_effect_phrase]

print(f'  ✓ Completed: {len(event_names_effect_phrase)} clusters')

print('\n' + '=' * 70)
print('✓ Effect Clustering Completed')
print('=' * 70)


[1/2] Role-based Clustering
  → K-Means clustering (k=100)...
    K-Means (k=100, n=3526)...
      ✓ 100 clusters
  → Generating cluster names...
  ✓ Completed: 100 clusters

[2/2] Phrase-based Clustering
  → K-Means clustering (k=100)...
    K-Means (k=100, n=4161)...
      ✓ 100 clusters
  → Generating cluster names...
  ✓ Completed: 100 clusters

✓ Effect Clustering Completed


In [ ]:
# ===== Clustering Results Statistics =====
print('\n' + '=' * 70)
print('Clustering Results Statistics')
print('=' * 70)

# Cause Statistics
cause_srl_clusters = df[df['cause_srl_cluster_id'] >= 0]['cause_srl_cluster_id'].nunique()
cause_phrase_clusters = df[df['cause_phrase_cluster_id'] >= 0]['cause_phrase_cluster_id'].nunique()

print(f'\nCause:')
print(f' - Role-based:  {cause_srl_clusters:3d} clusters, {(df["cause_srl_cluster_id"] >= 0).sum():4d} samples')
print(f' - Phrase-based: {cause_phrase_clusters:3d} clusters, {(df["cause_phrase_cluster_id"] >= 0).sum():4d} samples')

# Effect Statistics
effect_srl_clusters = df[df['effect_srl_cluster_id'] >= 0]['effect_srl_cluster_id'].nunique()
effect_phrase_clusters = df[df['effect_phrase_cluster_id'] >= 0]['effect_phrase_cluster_id'].nunique()

print(f'\nEffect:')
print(f' - Role-based:  {effect_srl_clusters:3d} clusters, {(df["effect_srl_cluster_id"] >= 0).sum():4d} samples')
print(f' - Phrase-based: {effect_phrase_clusters:3d} clusters, {(df["effect_phrase_cluster_id"] >= 0).sum():4d} samples')

print('\n' + '=' * 70)
print(f' - Role-based: {effect_srl_clusters} Clustering')
print(f' - Phrase-based: {effect_phrase_clusters} Clustering')

In [ ]:
# ===== Save results =====
print('\n' + '=' * 70)
print('Save Clustering Results')
print('=' * 70)

# Save as PKL
output_pkl = out_dir / 'event_results.pkl'
df.to_pickle(output_pkl)

---
## 8. Build Causal Narrative Network

The `CausalNetworkBuilder` constructs a directed graph where:
- **Nodes** = event clusters
- **Edges** = causal relationships between clusters
- **Edge weight** = frequency of the causal link

### `CausalNetworkBuilder` Methods

| Method | Description |
|--------|-------------|
| `add_causal_pair(cause_id, effect_id, cause_text, effect_text)` | Add one causal pair |
| `add_causal_pairs_batch(pairs, texts)` | Add multiple pairs |
| `build_from_dataframe(df, cause_col, effect_col)` | Build from DataFrame |
| `set_cluster_summaries(summaries)` | Set node labels/metadata |
| `get_edges(min_weight)` | Get edges as `CausalEdge` objects |
| `get_summary_stats()` | Network statistics |
| `export_to_csv(output_dir)` | Export nodes and edges to CSV |
| `export_to_graphml(output_path)` | Export to GraphML format |

### `CausalEdge` Fields

| Field | Type | Description |
|-------|------|-------------|
| `cause_cluster_id` | int | Source cluster (cause) |
| `effect_cluster_id` | int | Target cluster (effect) |
| `weight` | float | Edge weight (frequency) |
| `example_pairs` | List[Tuple] | Example (cause, effect) texts |
| `cause_label` / `effect_label` | str | Cluster labels |
| `to_dict()` | method | Convert to dictionary |

In [29]:
import pandas as pd

df_clustered = pd.read_pickle('output/event_results.pkl')
df_clustered.head()

,cause_text,effect_text,idx,sentence,doc_id,cause_srl,effect_srl,cause_srl_structure,effect_srl_structure,cause_srl_has_verb_arg,...,cause_srl_cluster_id,cause_srl_event,cause_phrase_cluster_id,cause_phrase_event,effect_srl_cluster_id,effect_srl_event,effect_phrase_cluster_id,effect_phrase_event,cause_span,effect_span
0,so many political mistakes being made,left years ago,0,"I made a great deal of money in Atlantic City,...",15748,"{""verbs"": [{""verb"": ""being"", ""description"": ""s...","{""verbs"": [{""verb"": ""left"", ""description"": ""[V...",v,v,False,...,-1,,1,gross incompetence,-1,,1,Was over,so many political mistakes being made,left years ago
1,other outlets take their place,Theyll all die together,5,Theyll all die together as other outlets take ...,28219,"{""verbs"": [{""verb"": ""take"", ""description"": ""[A...","{""verbs"": [{""verb"": ""ll"", ""description"": ""They...",a1v,v,True,...,71,other outlets take their place,-1,,-1,,67,die,other outlets take their place,Theyll all die together
2,their daughter being with Rosie--a true loser,Rosie 's new partner in love whose parents are...,17,I feel sorry for Rosie 's new partner in love ...,6173,"{""verbs"": [{""verb"": ""being"", ""description"": ""[...","{""verbs"": [{""verb"": ""are"", ""description"": ""Ros...",a1va2,v,True,...,2,being their daughter,-1,,-1,,15,Brazilian mills are celebrating,their daughter being with Rosie--a true loser,Rosie 's new partner in love whose parents are...
3,you dont deliver the goods,people will eventually catch on,20,"If you dont deliver the goods, people will eve...",6929,"{""verbs"": [{""verb"": ""do"", ""description"": ""you ...","{""verbs"": [{""verb"": ""will"", ""description"": ""pe...",v,v,False,...,-1,,70,This money,-1,,59,they will quickly go out of business,you dont deliver the goods,people will eventually catch on
4,we bought 15 Billion Dollars of Agriculture fr...,we would have more than 85 Billion Dollars lef...,24,....If we bought 15 Billion Dollars of Agricul...,31187,"{""verbs"": [{""verb"": ""bought"", ""description"": ""...","{""verbs"": [{""verb"": ""would"", ""description"": ""w...",a1va2,v,True,...,32,we bought 15 Billion Dollars of Agriculture,-1,,-1,,56,we would have more than 85 Billion Dollars lef...,we bought 15 Billion Dollars of Agriculture fr...,we would have more than 85 Billion Dollars lef...


In [30]:
# # Merge cause_event / effect_event columns: prioritize srl_event, otherwise use phrase_event
df_clustered['cause_event'] = df_clustered['cause_srl_event'].replace('', pd.NA).fillna(df_clustered['cause_phrase_event']).fillna('')
df_clustered['effect_event'] = df_clustered['effect_srl_event'].replace('', pd.NA).fillna(df_clustered['effect_phrase_event']).fillna('')

In [31]:
from causal_narrative.network import CausalNetworkBuilder

# Build network from clustering results
builder = CausalNetworkBuilder()

# Add causal pairs from clustered DataFrame
# # Use Role-based clustering results (preferred)
valid_rows = df_clustered[
    (df_clustered['cause_srl_event'].notna()) & 
    (df_clustered['effect_srl_event'].notna()) &
    (df_clustered['cause_srl_cluster_id'] >= 0) &
    (df_clustered['effect_srl_cluster_id'] >= 0)
].copy()

print(f'Building network with {len(valid_rows)} valid causal relationships...')

for _, row in valid_rows.iterrows():
    try:
        cause_id = int(row['cause_srl_cluster_id'])
        effect_id = int(row['effect_srl_cluster_id'])
        builder.add_causal_pair(
            cause_cluster_id=cause_id,
            effect_cluster_id=effect_id,
            cause_text=str(row.get('cause_span', '')),
            effect_text=str(row.get('effect_span', '')),
        )
    except (ValueError, TypeError) as e:
        continue

G = builder.graph

# Set node labels from cluster events
for node in G.nodes():
    # Find event name for this cluster from DataFrame
    if node in df_clustered['cause_srl_cluster_id'].values:
        event = df_clustered[df_clustered['cause_srl_cluster_id'] == node]['cause_srl_event'].iloc[0]
        G.nodes[node]['label'] = event[:50]  # Truncate long text
    elif node in df_clustered['effect_srl_cluster_id'].values:
        event = df_clustered[df_clustered['effect_srl_cluster_id'] == node]['effect_srl_event'].iloc[0]
        G.nodes[node]['label'] = event[:50]
    else:
        G.nodes[node]['label'] = f'Cluster {node}'

# Print summary
stats = builder.get_summary_stats()
print('\n=== Network Summary ===')
for k, v in stats.items():
    print(f'  {k}: {v}')


Building network with 1713 valid causal relationships...

=== Network Summary ===
  n_nodes: 100
  n_edges: 1380
  density: 0.1393939393939394
  avg_in_degree: 13.8
  avg_out_degree: 13.8
  max_in_degree: 33
  max_out_degree: 51
  is_weakly_connected: True
  n_weakly_connected_components: 1


In [32]:
# Get top edges by weight
edges = builder.get_edges(min_weight=2)
edges_sorted = sorted(edges, key=lambda e: e.weight, reverse=True)

print(f'=== Top 10 Causal Edges (weight >= 2) ===')
for i, edge in enumerate(edges_sorted[:10], 1):
    cause_label = G.nodes[edge.cause_cluster_id].get('label', str(edge.cause_cluster_id))
    effect_label = G.nodes[edge.effect_cluster_id].get('label', str(edge.effect_cluster_id))
    print(f'  {i:2d}. [{edge.weight:3.0f}x] "{cause_label}" -> "{effect_label}"')

=== Top 10 Causal Edges (weight >= 2) ===
   1. [  7x] "you love what you 're doing" -> "we bought 15 Billion Dollars of Agriculture"
   2. [  6x] "you work" -> "is it"
   3. [  5x] "losing a battle" -> "getting the Probers"
   4. [  5x] "growing economy" -> "'s It"
   5. [  4x] "listened to the flawed advice of paulkrugman" -> "vote Obama"
   6. [  4x] "listened to the flawed advice of paulkrugman" -> "Do Nothing"
   7. [  4x] "losing a battle" -> "They made a phony collusion with the Russians stor"
   8. [  4x] "came the Plague" -> "you work"
   9. [  4x] "is Obama" -> "getting the Probers"
  10. [  4x] "he ruled No Collusion in the long awaited $ 30,000" -> "tim_cook agreeing to expand operations in the U.S."


---
## 9. Network Filtering & Analysis

The `network.filters` module provides tools to refine and analyze the network.

### Filter Functions

| Function | Description | Key Parameters |
|----------|-------------|----------------|
| `filter_by_weight(G, min_weight)` | Remove light edges | `min_weight`: int, `inplace`: bool |
| `filter_by_degree(G, min_degree, degree_type)` | Remove low-degree nodes | `degree_type`: 'in'/'out'/'total' |
| `filter_top_k_nodes(G, k, criterion)` | Keep top-k nodes | `criterion`: 'degree'/'pagerank'/... |
| `apply_kcore(G, k)` | K-core decomposition | `k`: core number |

### Analysis Functions

| Function | Description | Returns |
|----------|-------------|----------|
| `compute_network_stats(G)` | Full statistics | `Dict` with centrality, connectivity, etc. |
| `get_hub_nodes(G, top_k, criterion)` | Top causes | `List[(node_id, score)]` |
| `get_authority_nodes(G, top_k, criterion)` | Top effects | `List[(node_id, score)]` |
| `find_cycles(G, max_length)` | Causal loops | `List[List[int]]` |
| `get_strongly_connected_components(G)` | SCC | `List[Set[int]]` |
| `get_weakly_connected_components(G)` | WCC | `List[Set[int]]` |

In [33]:
from causal_narrative.network import (
    filter_by_weight,
    filter_by_degree,
    filter_top_k_nodes,
    apply_kcore,
    compute_network_stats,
    get_hub_nodes,
    get_authority_nodes,
    find_cycles,
    get_strongly_connected_components,
    get_weakly_connected_components,
)
import networkx as nx

# Start from a copy of the full graph
G_filtered = G.copy()

# Remove self-loops
self_loops = list(nx.selfloop_edges(G_filtered))
G_filtered.remove_edges_from(self_loops)
print(f'Removed {len(self_loops)} self-loops')

# Step 1: Filter by edge weight
print(f'\nBefore weight filter: {G_filtered.number_of_nodes()} nodes, {G_filtered.number_of_edges()} edges')
G_filtered = filter_by_weight(G_filtered, min_weight=2)
print(f'After weight >= 2:    {G_filtered.number_of_nodes()} nodes, {G_filtered.number_of_edges()} edges')

# Step 2: Keep top-k nodes
G_viz = filter_top_k_nodes(G_filtered, k=40, criterion='degree')
print(f'After top-40 nodes:   {G_viz.number_of_nodes()} nodes, {G_viz.number_of_edges()} edges')

23:01:25 | INFO     | Filtered by weight >= 2: removed 1113 edges, 2 isolated nodes
23:01:25 | INFO     | Kept top 40 nodes by degree: removed 58 nodes


Removed 16 self-loops

Before weight filter: 100 nodes, 1364 edges
After weight >= 2:    98 nodes, 251 edges
After top-40 nodes:   40 nodes, 112 edges


In [34]:
# Compute comprehensive network statistics
net_stats = compute_network_stats(G_viz)

print('=== Network Statistics ===')
for k, v in net_stats.items():
    if isinstance(v, float):
        print(f'  {k:35s}: {v:.4f}')
    else:
        print(f'  {k:35s}: {v}')

23:01:28 | INFO     | Computing network statistics for graph with 40 nodes and 112 edges
23:01:28 | INFO     | Network statistics computed: 40 nodes, 112 edges, density=0.0718


=== Network Statistics ===
  n_nodes                            : 40
  n_edges                            : 112
  density                            : 0.0718
  avg_in_degree                      : 2.8000
  avg_out_degree                     : 2.8000
  avg_degree                         : 5.6000
  max_in_degree                      : 10
  max_out_degree                     : 12
  max_degree                         : 14
  is_strongly_connected              : False
  is_weakly_connected                : True
  n_strongly_connected_components    : 20
  n_weakly_connected_components      : 1
  avg_pagerank                       : 0.0250
  max_pagerank_node                  : 57
  avg_betweenness                    : 0.0389
  max_betweenness_node               : 57
  total_weight                       : 257
  avg_weight                         : 2.2946
  max_weight                         : 5


---
## 10. Visualization

The `viz` module provides multiple visualization options.

### Available Visualizations

| Function | Output | Interactive | Requirements |
|----------|--------|-------------|-------------|
| `plot_network_static()` | PNG/SVG | No | matplotlib |
| `plot_network_interactive_pyvis()` | HTML | Yes | pyvis |
| `plot_network_interactive_plotly()` | HTML | Yes | plotly |

### `plot_network_static()` Parameters

| Parameter | Type | Default | Description |
|-----------|------|---------|-------------|
| `G` | nx.DiGraph | (required) | Graph to visualize |
| `output_path` | Path | (required) | Output PNG/SVG path |
| `layout` | str | `'spring'` | `'spring'`, `'kamada_kawai'`, `'circular'` |
| `figsize` | tuple | `(16, 12)` | Figure size in inches |
| `dpi` | int | 300 | Resolution |
| `node_size_attr` | str | `'degree'` | `'degree'`, `'pagerank'`, `'betweenness'` |
| `edge_width_attr` | str | `'weight'` | Edge attribute for width |
| `color_by_community` | bool | True | Color by Louvain community |
| `label_top_k` | int | 20 | Label only top K nodes |
| `title` | str | None | Plot title |

In [37]:
from causal_narrative.viz import (
    plot_network_interactive_pyvis,
    plot_network_interactive_plotly,
)

# Interactive visualization with pyvis
pyvis_path = out_dir / 'causal_network_pyvis.html'
plot_network_interactive_pyvis(
    G_viz,
    output_path=pyvis_path,
    title='Trump Tweet Causal Narrative Network',
    height='800px',
    width='100%',
    notebook=False,
    top_n=100,  # Show only top 100 nodes by degree
    font_size=20,  # Larger font size for better readability
)
print(f'Interactive visualization saved to: {pyvis_path}')

23:01:41 | INFO     | Saved interactive pyvis visualization to output/causal_network_pyvis.html


Interactive visualization saved to: output/causal_network_pyvis.html
